In [3]:
import os
import json
import numpy as np
from openai import OpenAI
from google.colab import userdata
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))

In [5]:
OPENAI_EMBEDDING_MODEL='text-embedding-3-small'
# embedding_model = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
embedding_model = OPENAI_EMBEDDING_MODEL
embedding_dim = 1536  # Number of dimensions in the embedding vector
print(f"Using OpenAI model: {embedding_model}")
print(f"Embedding dimension: {embedding_dim}")


Using OpenAI model: text-embedding-3-small
Embedding dimension: 1536


In [7]:
# =============================================================================
# LOAD DATA: Support Tickets
# =============================================================================
print("\nLoading support tickets...")
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"Loaded {len(tickets)} support tickets")


Loading support tickets...
Loaded 20 support tickets


In [8]:
# Display sample ticket - understand what we're working with
print("\n" + "="*80)
print("SAMPLE TICKET:")
print("="*80)
sample = tickets[0]
print(f"ID: {sample['ticket_id']}")
print(f"Title: {sample['title']}")
print(f"Description: {sample['description'][:200]}...")
print(f"Category: {sample['category']}")
print(f"Priority: {sample['priority']}")


SAMPLE TICKET:
ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment....
Category: Authentication
Priority: High


In [9]:
print("\n" + "="*80)
print("PART 1: Generating Embeddings")
print("="*80)

ticket_texts = [
    f"{ticket['title']}. {ticket['description']}"
    for ticket in tickets
]
print(f"\nticket_texts: '{str(ticket_texts)}'")


print("\nGenerating embeddings for all tickets...")
response = client.embeddings.create(input=ticket_texts, model=embedding_model)

# Convert API response to NumPy array for mathematical operations
# Shape: (num_tickets, 1536) - each row is one ticket's embedding
embeddings = np.array([data.embedding for data in response.data])
print(f"✓ Generated embeddings with shape: {embeddings.shape}")
print(f"  ({len(tickets)} tickets × {embedding_dim} dimensions)")


PART 1: Generating Embeddings

ticket_texts: '["Users unable to log in after password reset. Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.", "Database connection timeout in production. Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.", "Payment processing fails for international cards. Customers using non-US credit cards receiving 'Payment declined' errors. Payment gateway returns error code 402. Issue only affects cards issued outside United States.", 'Mobile app crashes on iOS 17. App crashes immediately on launch for users who upgraded to iOS 17. Crash reports show EXC_BAD_ACCESS in networking module. Affects approximately 30% of iOS user base.', 'Memory leak in background worker process. Background job processor showing ste

In [10]:
print(f"\nFirst 10 values of embedding for ticket 1:")
print(embeddings[0][:10])
print("  (These 1536 numbers encode the semantic meaning of the text)")


First 10 values of embedding for ticket 1:
[-0.00804138  0.05169678  0.05493164  0.02616882 -0.01286316 -0.02456665
 -0.00462341  0.06231689 -0.00145721  0.00541687]
  (These 1536 numbers encode the semantic meaning of the text)


In [11]:
print("\n" + "="*80)
print("PART 2: Computing Similarity Scores")
print("="*80)

# Create a search query - this is what a user might type
query = "Users can't login after changing password"
print(f"\nSearch Query: '{query}'")

# -----------------------------------------------------------------------------
# Generate embedding for the query
# -----------------------------------------------------------------------------

query_response = client.embeddings.create(input=[query], model=embedding_model)
query_embedding = np.array([query_response.data[0].embedding])
print(f"Query embedding shape: {query_embedding.shape}")  # (1,


PART 2: Computing Similarity Scores

Search Query: 'Users can't login after changing password'
Query embedding shape: (1, 1536)


In [12]:
# -----------------------------------------------------------------------------
# Compute cosine similarity between query and ALL tickets
# -----------------------------------------------------------------------------

similarities = cosine_similarity(query_embedding, embeddings)[0]
print(f"\nComputed similarities look like {similarities}")
print(f"\nComputed similarity scores for {len(similarities)} tickets")
print(f"Similarity range: [{similarities.min():.4f}, {similarities.max():.4f}]")


Computed similarities look like [0.62178151 0.25581473 0.25077553 0.23661709 0.1417756  0.42162
 0.18163477 0.28526456 0.19952614 0.32674544 0.45567722 0.22175599
 0.16222309 0.37893797 0.17652577 0.42190561 0.22346326 0.21763571
 0.22053875 0.35610801]

Computed similarity scores for 20 tickets
Similarity range: [0.1418, 0.6218]


In [14]:
# ============================================================================
# PART 3: Retrieve Most Similar Tickets
# ============================================================================

print("\n" + "="*80)
print("PART 3: Finding Most Similar Tickets")
print("="*80)

# Get top-2 most similar tickets
top_k = 2

top_indices = np.argsort(similarities)[::-1][:top_k]

print(f"\nTop {top_k} most similar tickets to query: '{query}'")
print(f"\ntop_indices: '{top_indices}'")
print("-" * 80)

for rank, idx in enumerate(top_indices, 1):
    ticket = tickets[idx]
    score = similarities[idx]

    print(f"\n#{rank} - Similarity: {score:.4f}")
    print(f"Ticket ID: {ticket['ticket_id']}")
    print(f"Title: {ticket['title']}")
    print(f"Category: {ticket['category']} | Priority: {ticket['priority']}")
    print(f"Description: {ticket['description'][:150]}...")



PART 3: Finding Most Similar Tickets

Top 2 most similar tickets to query: 'Users can't login after changing password'

top_indices: '[ 0 10]'
--------------------------------------------------------------------------------

#1 - Similarity: 0.6218
Ticket ID: TICK-001
Title: Users unable to log in after password reset
Category: Authentication | Priority: High
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent sec...

#2 - Similarity: 0.4557
Ticket ID: TICK-011
Title: SSO authentication broken after upgrade
Category: Authentication | Priority: Critical
Description: Single Sign-On integration with corporate SAML provider failing. Users redirected to SSO portal but then receive 'Invalid SAML response' error when re...
